# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> dataset using the `mlcroissant` library, referencing all structures by their `@id` field as per the Croissant specification.

### Dataset Source
The dataset is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step fetches the dataset's structure, available record sets, and schema directly from the Croissant schema definition.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n\nCitation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Let's review available RecordSets, their fields, and columns using their `@id` values.

In [ ]:
# List all available RecordSets and their Fields
record_sets = metadata.record_sets
print(f"Number of RecordSets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet ID: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    # Fields
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields (@id):")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    print()

### Example: Previewing records of a RecordSet
We'll print the first few entries from a selected record set using its `@id`.

In [ ]:
# For demonstration, select the first RecordSet by @id
record_set_id = record_sets[0]['@id'] if record_sets else None
if record_set_id:
    print(f"Records from RecordSet {record_set_id}:")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No RecordSets available in the metadata.")

## 3. Data Extraction
Let's load available data from each record set into pandas DataFrames, using only their `@id` fields for reference.

In [ ]:
# Extract data from all record sets
dataframes = {}
all_record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in all_record_set_ids:
    # Get records as list of dict for this record set
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet {rs_id}. Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for RecordSet {rs_id}.")
    except Exception as e:
        print(f"Could not load RecordSet {rs_id}: {e}")

# Example: Show first 5 records from the primary record set
if record_set_id in dataframes:
    print(f"\nColumns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We select a numeric field by its `@id`. Steps:
- Filter records where the value for the numeric field exceeds a threshold.
- Normalize the numeric field.
- Optionally, group by a categorical field.

Be sure to inspect the columns and their corresponding `@id`s printed in the previous step to customize this section as needed.

In [ ]:
# --- Adjust the following variables as appropriate based on your dataset overview ---
# For demonstration, pick numeric_field_id and group_field_id present in the dataset

import numpy as np
primary_rs_id = record_set_id

if primary_rs_id in dataframes:
    df = dataframes[primary_rs_id]
    # Attempt to auto-detect a numeric field
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]
    else:
        numeric_field_id = df.columns[0] # fallback (likely not numeric!)

    # Attempt to auto-detect a group/categorical field (uses second column if any)
    if len(df.columns) > 1:
        group_field = df.columns[1]
    else:
        group_field = df.columns[0]

    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else None
    print(f"Using numeric_field_id = {numeric_field_id}\nUsing group_field = {group_field}\nUsing threshold = {threshold}")

    # Filter (only if we have numeric values and a valid threshold)
    if threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if it's not numeric
        if group_field in filtered_df.columns and not np.issubdtype(filtered_df[group_field].dtype, np.number):
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped average {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Let's plot a distribution of the numeric field used in the EDA and a bar plot if grouping was done.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id in dataframes:
    df = dataframes[primary_rs_id]
    if numeric_field_id in df.columns and np.issubdtype(df[numeric_field_id].dtype, np.number):
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

    # If we have a group field and grouping
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f'Average {numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR<sup>2</sup> mass spectrometry dataset via the `mlcroissant` library, referencing schema entities and data by their Croissant `@id`. We:

- Loaded and previewed dataset metadata and schema
- Dynamically extracted all available record sets and their fields by `@id`
- Loaded records into pandas DataFrames using these IDs
- Applied basic exploratory analysis and normalization to a representative numeric field
- Visualized field distributions and group differences

Adjust field IDs and groupings as needed to further interrogate domain-specific properties of this highly structured Croissant-compliant dataset.